In [1]:
import pandas as pd
import numpy as np
from semopy import Model, Optimizer
import semopy.plot as semplot
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load your data
df = pd.read_excel(r"C:\Users\pc726\OneDrive\Desktop\New folder\survey_data.xlsx")

In [3]:
df.columns

Index(['Timestamp', '1. Gender', '2. Age Group', '3. Qualification',
       '4. Professional Year ',
       '5. How would you rate your digital literacy skills?',
       '6. Which social media platforms do you use for academic purposes? (Select all that apply)',
       '7. How frequently do you use social media for academic purposes?',
       '8. What type of academic content do you access on social media? (Select all that apply)',
       '9. Do you think social media helps improve your academic performance?',
       '10. How does social media benefit your medical education? (Select all that apply)',
       '11. Do you think social media is more effective than traditional learning methods (books, lectures)?',
       '12. Why do you prefer social media over traditional sources? (Select all that apply)',
       '13. What motivates you to share academic content on social media? (Select all that apply)',
       '14. What challenges do you face while using social media for medical education

In [4]:
# ----------------------------
# DATA PREPROCESSING
# ----------------------------

# 1. SocialMediaUse construct
# Count platforms used
df['platform_count'] = df['6. Which social media platforms do you use for academic purposes? (Select all that apply)'].apply(
    lambda x: len(str(x).split(',')) if pd.notna(x) else 0
)

# Frequency of use encoding
freq_map = {'1-2 Hours': 1, '2-3 Hours': 2, '3-4 Hours': 3, 'More than four': 4}
df['frequency_encoded'] = df['7. How frequently do you use social media for academic purposes?'].map(freq_map)

# 2. DigitalLiteracy construct
digital_map = {'Beginner': 1, 'Intermediate': 2, 'Advanced': 3}
df['digital_literacy'] = df['5. How would you rate your digital literacy skills?'].map(digital_map)

# 3. AcademicBenefit construct
benefit_text = '10. How does social media benefit your medical education? (Select all that apply)'
df['helps_exam'] = df[benefit_text].apply(lambda x: 1 if 'Helps in exam preparation' in str(x) else 0)
df['expert_access'] = df[benefit_text].apply(lambda x: 1 if 'Provides access to expert knowledge' in str(x) else 0)
df['enhances_collab'] = df[benefit_text].apply(lambda x: 1 if 'Enhances collaboration with peers' in str(x) else 0)

# 4. MisinformationAwareness construct
df['aware_misinfo'] = df['15. Are you aware for misinformation and disinformation on social media? '].apply(
    lambda x: 1 if 'Yes' in str(x) else 0
)
df['active_verify'] = df['15. Are you aware for misinformation and disinformation on social media? '].apply(
    lambda x: 1 if 'actively verify' in str(x).lower() else 0
)

# 5. VerificationBehavior construct
verify_text = '16. How do you verify the credibility of academic content on social media? (select all that employ) '
df['checks_source'] = df[verify_text].apply(lambda x: 1 if 'Checking the source' in str(x) else 0)
df['compare_multiple'] = df[verify_text].apply(lambda x: 1 if 'Comparison information from multiple sources' in str(x) else 0)
df['peer_reviewed'] = df[verify_text].apply(lambda x: 1 if 'Looking for peer-reviewed refrences' in str(x) else 0)

# 6. TrainingInterest construct
training_map = {'Yes': 1, 'Maybe': 0.5, 'No': 0}
df['training_interest'] = df['18. Would you be interested in a training program on using social media for academic and medical learning?'].map(training_map)

# Create final dataset for SEM
sem_data = df[[
    'platform_count', 'frequency_encoded',
    'digital_literacy',
    'helps_exam', 'expert_access', 'enhances_collab',
    'aware_misinfo', 'active_verify',
    'checks_source', 'compare_multiple', 'peer_reviewed',
    'training_interest'
]].copy()

# Handle missing values
sem_data = sem_data.dropna()

# Standardize continuous variables (optional but recommended)
scaler = StandardScaler()
continuous_vars = ['platform_count', 'frequency_encoded', 'digital_literacy', 'training_interest']
sem_data[continuous_vars] = scaler.fit_transform(sem_data[continuous_vars])

print(f"Final dataset shape: {sem_data.shape}")
print("\nSample of preprocessed data:")
print(sem_data.head())

# ----------------------------
# STRUCTURAL EQUATION MODELING
# ----------------------------

# Define the SEM model
# model_spec = """
# # Measurement model
# SocialMediaUse =~ platform_count + frequency_encoded
# DigitalLiteracy =~ digital_literacy
# AcademicBenefit =~ helps_exam + expert_access + enhances_collab
# MisinformationAwareness =~ aware_misinfo + active_verify
# VerificationBehavior =~ checks_source + compare_multiple + peer_reviewed
# TrainingInterest =~ training_interest

# # Structural model
# AcademicBenefit ~ SocialMediaUse + DigitalLiteracy
# VerificationBehavior ~ DigitalLiteracy + MisinformationAwareness + TrainingInterest
# MisinformationAwareness ~ SocialMediaUse + DigitalLiteracy
# DigitalLiteracy ~ TrainingInterest

# # Covariances (if needed)
# SocialMediaUse ~~ DigitalLiteracy
# """

# Revised Model Specification
model_spec = """
    # ===== MEASUREMENT MODEL =====
    SocialMediaUse =~ platform_count + frequency_encoded
    DigitalLiteracy =~ digital_literacy
    TrainingInterest =~ training_interest
    AcademicBenefit =~ helps_exam + expert_access + enhances_collab
    MisinformationAwareness =~ aware_misinfo + active_verify
    VerificationBehavior =~ checks_source

    # ===== STRUCTURAL MODEL (SIMPLIFIED) =====
    # Only keeping significant and theoretically meaningful paths
    AcademicBenefit ~ DigitalLiteracy
    VerificationBehavior ~ MisinformationAwareness
    MisinformationAwareness ~ SocialMediaUse

    # ===== COVARIANCES (ADDITIONAL FOR BETTER FIT) =====
    # Among exogenous variables
    DigitalLiteracy ~~ SocialMediaUse
    DigitalLiteracy ~~ TrainingInterest
    SocialMediaUse ~~ TrainingInterest
    
    # Among indicator residuals (based on modification indices)
    helps_exam ~~ enhances_collab
    expert_access ~~ enhances_collab
    aware_misinfo ~~ active_verify
    
    # Additional error covariances for better fit
    helps_exam ~~ expert_access
    platform_count ~~ frequency_encoded
    digital_literacy ~~ training_interest
    
    # Residual covariances for endogenous variables
    AcademicBenefit ~~ TrainingInterest
    AcademicBenefit ~~ SocialMediaUse
    MisinformationAwareness ~~ DigitalLiteracy
"""

# Initialize and fit the model
model = Model(model_spec)
try:
    # Fit the model
    result = model.fit(sem_data)
    
    # Display results
    print("\n" + "="*50)
    print("SEM MODEL RESULTS")
    print("="*50)
    
    # Model fit indices
    print("\nMODEL FIT INDICES:")
    fit_stats = model.inspect()
    print(fit_stats)
    
    # Parameter estimates
    print("\nPARAMETER ESTIMATES:")
    estimates = model.inspect(std_est=True)
    print(estimates.round(3))
    
    # Goodness of fit measures
    print("\nGOODNESS OF FIT MEASURES:")
    gfi = model.gfi()
    agfi = model.agfi()
    rmsea = model.rmsea()
    cfi = model.cfi()
    
    print(f"GFI: {gfi:.3f}")
    print(f"AGFI: {agfi:.3f}")
    print(f"RMSEA: {rmsea:.3f}")
    print(f"CFI: {cfi:.3f}")
    
    # Interpretation of fit indices
    print("\nFIT INTERPRETATION:")
    if rmsea < 0.06 and cfi > 0.95:
        print("✓ Excellent model fit!")
    elif rmsea < 0.08 and cfi > 0.90:
        print("✓ Acceptable model fit")
    else:
        print("✗ Poor model fit - consider model modifications")
    
    # Plot the model (optional)
    try:
        fig, ax = plt.subplots(figsize=(12, 8))
        semplot.visualize(model, "model_diagram.png", plot_params=True)
        print("\nModel diagram saved as 'model_diagram.png'")
    except:
        print("Could not generate model diagram - check graphviz installation")
    
    # ----------------------------
    # HYPOTHESIS TESTING
    # ----------------------------
    print("\n" + "="*50)
    print("HYPOTHESIS TESTING")
    print("="*50)
    
    # Extract significant paths
    sig_paths = estimates[(estimates['op'] == '~') & (estimates['p-value'] < 0.05)]
    
    if not sig_paths.empty:
        print("Significant paths (p < 0.05):")
        for idx, row in sig_paths.iterrows():
            effect = "Positive" if row['Estimate'] > 0 else "Negative"
            print(f"{row['lval']} -> {row['rval']}: {effect} effect (β = {row['Estimate']:.3f}, p = {row['p-value']:.3f})")
    else:
        print("No significant paths found at p < 0.05")
        
except Exception as e:
    print(f"Error fitting model: {e}")
    print("Trying alternative estimation method...")
    
    # Try with different optimizer
    try:
        opt = Optimizer(model_spec)
        opt.optimize(sem_data)
        result = opt.fit(sem_data)
        print("Model fitted successfully with alternative method!")
        print(opt.inspect())
    except Exception as e2:
        print(f"Still unable to fit model: {e2}")

# ----------------------------
# MODEL MODIFICATION (if needed)
# ----------------------------

# If model fit is poor, you can try modifications:
print("\n" + "="*50)
print("MODEL MODIFICATION SUGGESTIONS")
print("="*50)
print("If model fit is poor, consider:")
print("1. Adding error covariances between related indicators")
print("2. Removing non-significant paths")
print("3. Adding direct effects based on modification indices")
print("4. Trying different estimation methods (ML, WLS, etc.)")

# Example of modification indices (if model converged)
try:
    if 'model' in locals():
        mi = model.modification_indices(sem_data)
        print("\nTop modification indices:")
        print(mi[mi['mi'] > 3.84].sort_values('mi', ascending=False).head(10))
except:
    pass

# ----------------------------
# SAVE RESULTS
# ----------------------------

# Save parameter estimates to CSV
try:
    estimates.to_csv('sem_parameter_estimates.csv')
    print("\nParameter estimates saved to 'sem_parameter_estimates.csv'")
except:
    pass

print("\nSEM analysis completed!")

Final dataset shape: (427, 12)

Sample of preprocessed data:
   platform_count  frequency_encoded  digital_literacy  helps_exam  \
0       -0.599937           0.724091         -0.163155           0   
1       -0.599937          -0.204398         -0.163155           1   
2       -1.221717          -0.204398          1.457014           0   
3       -1.221717          -0.204398         -0.163155           1   
4       -1.221717          -0.204398         -0.163155           0   

   expert_access  enhances_collab  aware_misinfo  active_verify  \
0              1                0              1              0   
1              1                0              1              1   
2              0                0              1              1   
3              1                1              1              0   
4              1                0              1              1   

   checks_source  compare_multiple  peer_reviewed  training_interest  
0              1                 0          

                       lval  op                     rval      Estimate  \
0           AcademicBenefit   ~          DigitalLiteracy  1.162136e+00   
1      VerificationBehavior   ~  MisinformationAwareness  2.238644e+00   
2   MisinformationAwareness   ~           SocialMediaUse  1.201478e-01   
3            platform_count   ~           SocialMediaUse  1.000000e+00   
4         frequency_encoded   ~           SocialMediaUse  1.565714e-01   
5          digital_literacy   ~          DigitalLiteracy  1.000000e+00   
6         training_interest   ~         TrainingInterest  1.000000e+00   
7                helps_exam   ~          AcademicBenefit  1.000000e+00   
8             expert_access   ~          AcademicBenefit  1.149466e+00   
9           enhances_collab   ~          AcademicBenefit  2.526372e+00   
10            aware_misinfo   ~  MisinformationAwareness  1.000000e+00   
11            active_verify   ~  MisinformationAwareness  5.411702e+00   
12            checks_source   ~     Ve

In [5]:
import semopy
stat=semopy.calc_stats(model)
print(stat)

       DoF  DoF Baseline       chi2  chi2 p-value  chi2 Baseline       CFI  \
Value   20            45  38.805763      0.007048     238.268209  0.902696   

            GFI      AGFI       NFI       TLI     RMSEA       AIC        BIC  \
Value  0.837134  0.633552  0.837134  0.781066  0.046981  69.81824  211.80568   

        LogLik  
Value  0.09088  


In [6]:
import tabulate
print(tabulate.tabulate(stat, headers='keys', tablefmt='psql'))

+-------+-------+----------------+---------+----------------+-----------------+----------+----------+----------+----------+----------+-----------+---------+---------+----------+
|       |   DoF |   DoF Baseline |    chi2 |   chi2 p-value |   chi2 Baseline |      CFI |      GFI |     AGFI |      NFI |      TLI |     RMSEA |     AIC |     BIC |   LogLik |
|-------+-------+----------------+---------+----------------+-----------------+----------+----------+----------+----------+----------+-----------+---------+---------+----------|
| Value |    20 |             45 | 38.8058 |     0.00704801 |         238.268 | 0.902696 | 0.837134 | 0.633552 | 0.837134 | 0.781066 | 0.0469814 | 69.8182 | 211.806 |  0.09088 |
+-------+-------+----------------+---------+----------------+-----------------+----------+----------+----------+----------+----------+-----------+---------+---------+----------+


In [7]:
pd.DataFrame(stat).to_csv('sem_fit_statistics.csv')